# Step 4 — LLM Judge Evaluation (four-method comparison)

Score each filter method's Skip-gram recommendations with a local LLM judge (Ollama / `llama3`). For a shared set of evaluation anchors, every method's top-10 neighbours are judged as acceptable substitutes or not, then aggregated into two headline metrics per method: **Precision@10** and a **novel-substitute rate**. One judge, one prompt, one loop — only the input (which filter produced the recommendations) differs, so any score gap is attributable to the method.

The department-restricted disparity variant is excluded (a negative result).

**Inputs** (in `outputs/`)
- `evaluation_sample.json` — the shared all-four-coverage anchor set (from `03_skipgram.ipynb`)
- `skipgram/{method}/{value}/top10.jsonl` — each method's top-10 recommendations
- `products_text.csv` — product names
- `embeddings.npy`, `product_id_to_index.json` — Step 1 vectors (novel-substitute check)

**Outputs** (to `outputs/`)
- `llm_judge/results.jsonl` — one record per judged (anchor, method, candidate); append-only, resume-supported
- `llm_judge/summary.json` — the per-method headline metrics
- `llm_judge/spotcheck.txt` — 30 sampled judgments for a manual reliability read-through

**Requires** a local Ollama server with the `llama3` model pulled. Author-only; run it yourself.

## 1. Imports, paths, config

In [1]:
import json, time, random
from pathlib import Path
import pandas as pd
import numpy as np
import ollama

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT_DIR = ROOT / "outputs"
STEP1_DIR = OUT_DIR / "step1"
SKIPGRAM_DIR = OUT_DIR / "skipgram"
LLM_JUDGE_DIR = OUT_DIR / "llm_judge"
LLM_JUDGE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "llama3"
RANDOM_SEED = 42
MAX_RETRIES = 2
EVAL_SAMPLE_SIZE = 100   # upper bound on anchors actually judged
METHODS = ["constant_theta", "pct_dropoff", "kneedle", "disparity"]

# Each method maps to its value-labeled subfolder under skipgram/.
METHOD_DIRS = {
    "constant_theta": "constant_theta/theta_0.70",
    "pct_dropoff": "pct_dropoff/drop_0.10",
    "kneedle": "kneedle/sensitivity_1.0",
    "disparity": "disparity/alpha_0.05",
}

RESULTS_PATH = LLM_JUDGE_DIR / "results.jsonl"
SUMMARY_PATH = LLM_JUDGE_DIR / "summary.json"
SPOTCHECK_PATH = LLM_JUDGE_DIR / "spotcheck.txt"

# Stress-test anchors — always judged, regardless of the random draw (see §3).
STRESS_TEST_IDS = {
    4210:  "Whole Milk",
    6347:  "Unsweetened Almond Milk",
    651:   "Organic Salted Butter",
    16268: "Vegan Butter",
    6104:  "Whole Milk Plain Yogurt",
    920:   "Coconut Yogurt",
}

print("Outputs dir:", OUT_DIR)

Outputs dir: /Users/linnhtet/Developer/bachelor-thesis/outputs


## 2. The corrected judge prompt, and the judging mechanics

The one substantive change from the reference judge is the `SYSTEM_PROMPT`. The original prompt let the model misclassify organic-vs-non-organic pairs as directional (e.g. "Organic Distilled White Vinegar" vs "Distilled White Vinegar" -> `restricted_item="original"` -> rejected), because the organic/certification carve-out lived in its own paragraph AFTER the classification instruction, by which point the model had often already committed. Here the carve-out is merged into the first paragraph, next to the real restricted-attribute examples, with a worked example added. Everything else (flavor guidance, brand guidance, JSON schema) is unchanged.

`build_user_prompt`, `resolve_verdict`, and `judge_pair` are reused exactly from the reference — the directional-pair resolution (candidate-restricted is acceptable, original-restricted is not) is already correct and untouched.

In [2]:
SYSTEM_PROMPT = """You are a grocery shopping assistant helping evaluate whether one \
product could reasonably substitute another when the original is unavailable.

You will be given two product names: the ORIGINAL product a customer wants, and a \
CANDIDATE product that might be offered instead if the original is out of stock.

First, decide whether this pair differs specifically by ONE product being a reduced, \
restricted, or "free-from" version of an attribute the OTHER has in its standard \
form. Examples: fat content (fat-free vs whole), caffeine (decaf vs regular), sugar \
(sugar-free vs regular), sweetness (unsweetened vs sweetened), alcohol \
(non-alcoholic vs regular), gluten (gluten-free vs regular). This does NOT include \
"organic", "natural", "cage-free", or similar certification or preference labels, \
even though they can look similar at a glance — treat these the same as a flavor \
or brand difference: usually an acceptable substitute in either direction, since \
most shoppers treat them as a preference rather than a requirement. Example: \
"Organic Whole Milk" vs "Whole Milk" is NOT a restricted-attribute pair \
(is_directional_pair: false), the same as two different brands of the same \
product would be.

If this pair IS this kind of restricted-attribute pair, set "is_directional_pair" \
to true, and set "restricted_item" to either "original" or "candidate", whichever \
one is the reduced/restricted/free-from version. Your "is_substitute" field will be \
ignored for this kind of pair, so just make your best guess.

If this pair is NOT this kind of restricted-attribute pair (a flavor difference, \
brand difference, or genuinely different product), set "is_directional_pair" to \
false, "restricted_item" to null, and judge "is_substitute" directly using this \
guidance:
- Different flavors, scents, or varieties of the exact same type of item (two \
flavors of ice cream, juice, soda, chips, etc.) are ALWAYS acceptable substitutes \
for each other. Only say false here if the candidate is a genuinely different \
product, not just a different flavor.
- Different brands of the same type of product are usually acceptable substitutes.
- Say false mainly when the candidate is a genuinely different product category or \
use case.

Respond with a JSON object only, in exactly this format, with no other text:
{"reason": "one short phrase, under 10 words", "is_directional_pair": true or false, \
"restricted_item": "original" or "candidate" or null, "is_substitute": true or false}"""

def build_user_prompt(anchor, candidate):
    return (
        f'Original product: "{anchor}"\n'
        f'Candidate product: "{candidate}"\n\n'
        f'If a customer wanted to buy "{anchor}" but it was out of stock, '
        f'would "{candidate}" be an acceptable substitute?'
    )

def resolve_verdict(parsed):
    """For restricted-attribute pairs, apply the rule ourselves in code, so it's
    never at the mercy of the model getting the branch backwards."""
    if parsed.get("is_directional_pair") and parsed.get("restricted_item") in ("original", "candidate"):
        # Acceptable only if the CANDIDATE is the restricted one (offering someone
        # who wanted standard a slightly-reduced version).
        # Not acceptable if the ORIGINAL is restricted (offering someone who
        # specifically wanted restricted the plain standard version instead).
        return parsed["restricted_item"] == "candidate"
    return parsed["is_substitute"]

# ---- Ollama call with defensive parsing ----
def judge_pair(anchor, candidate, retries=MAX_RETRIES):
    for attempt in range(retries + 1):
        try:
            response = ollama.chat(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(anchor, candidate)},
                ],
                format="json",
                options={"temperature": 0},
            )
            content = response["message"]["content"].strip()
            content = content.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            parsed = json.loads(content)

            if not isinstance(parsed.get("is_substitute"), bool):
                raise ValueError(f"is_substitute not a bool: {parsed}")

            final_verdict = resolve_verdict(parsed)

            return {
                "is_substitute": final_verdict,
                "raw_is_substitute": parsed["is_substitute"],
                "is_directional_pair": parsed.get("is_directional_pair", False),
                "restricted_item": parsed.get("restricted_item"),
                "reason": str(parsed.get("reason", ""))[:200],
                "parse_error": False,
            }
        except Exception as e:
            if attempt == retries:
                return {"is_substitute": None, "raw_is_substitute": None,
                         "is_directional_pair": None, "restricted_item": None,
                         "reason": f"PARSE_FAILED: {e}", "parse_error": True}
            time.sleep(0.5)

## 3. Load the four methods' top-10s and the shared evaluation sample

Draw the anchors to judge from `evaluation_sample.json`'s `all_four_coverage` (the anchors every method has a vector for, so the comparison is apples-to-apples), capped at `EVAL_SAMPLE_SIZE` with a fixed seed, always including the six stress-test anchors. Then load each method's `skipgram/{method}/{value}/top10.jsonl`, resolve the IDs to names via `products_text.csv`, and build one long-format dataframe: one row per (anchor, method, ranked candidate). Anchors with fewer than 10 candidates simply contribute however many they have.

In [3]:
# ---- Product names ----
products_text = pd.read_csv(STEP1_DIR / "products_text.csv")
id_to_name = dict(zip(products_text["product_id"], products_text["product_name"]))


def name(pid):
    return id_to_name.get(int(pid), f"?{pid}")


# ---- Anchors to judge: capped draw from all-four coverage + the six stress anchors ----
with open(SKIPGRAM_DIR / "evaluation_sample.json") as f:
    evaluation_sample = json.load(f)
all_four = [int(p) for p in evaluation_sample["all_four_coverage"]]

sampler = random.Random(RANDOM_SEED)
k = min(EVAL_SAMPLE_SIZE, len(all_four))
judged_anchors = set(sampler.sample(all_four, k))
judged_anchors.update(STRESS_TEST_IDS)   # always include the stress anchors

# ---- Build the long-format table across all four methods ----
long_rows = []
for method in METHODS:
    path = SKIPGRAM_DIR / METHOD_DIRS[method] / "top10.jsonl"
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            aid = int(rec["anchor_id"])
            if aid not in judged_anchors:
                continue
            for rank, cid in enumerate(rec["neighbours"], start=1):   # however many exist (<=10)
                cid = int(cid)
                long_rows.append({
                    "anchor_id": aid,
                    "anchor_name": name(aid),
                    "method": method,
                    "rank": rank,
                    "candidate_id": cid,
                    "candidate_name": name(cid),
                })

long_df = pd.DataFrame(long_rows)

n_pairs = len(long_df)
n_anchors = long_df["anchor_id"].nunique()
print(f"anchors to judge : {n_anchors:,} (pool cap {EVAL_SAMPLE_SIZE} + stress anchors)")
print(f"total pairs      : {n_pairs:,} (anchors x methods x candidates-per-anchor)")
print(f"pairs per method :")
print(long_df.groupby("method").size().to_string())
lo, hi = n_pairs * 1 / 60, n_pairs * 2 / 60
print(f"\nrough time @1-2s/call: ~{lo:.0f}-{hi:.0f} min "
      f"(informational only — the loop below has resume support and can be stopped/continued)")

anchors to judge : 101 (pool cap 100 + stress anchors)
total pairs      : 4,040 (anchors x methods x candidates-per-anchor)
pairs per method :
method
constant_theta    1010
disparity         1010
kneedle           1010
pct_dropoff       1010

rough time @1-2s/call: ~67-135 min (informational only — the loop below has resume support and can be stopped/continued)


## 4. The judging loop (resume-supported)

Same append-and-flush resume pattern as the reference, keyed on `(anchor_id, method, candidate_id)` so a stopped run continues where it left off. Each row is judged by `judge_pair`, written immediately, and flushed.

In [4]:
# ---- Resume: skip (anchor_id, method, candidate_id) triples already judged ----
already_done = set()
if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        for line in f:
            r = json.loads(line)
            already_done.add((r["anchor_id"], r["method"], r["candidate_id"]))
    print(f"Resuming: {len(already_done):,} pairs already judged, will skip those.\n")

# ---- Main loop ----
with open(RESULTS_PATH, "a") as f_out:
    for i, row in long_df.iterrows():
        key = (int(row["anchor_id"]), row["method"], int(row["candidate_id"]))
        if key in already_done:
            continue

        result = judge_pair(row["anchor_name"], row["candidate_name"])
        record = {
            "anchor_id": int(row["anchor_id"]),
            "anchor_name": row["anchor_name"],
            "method": row["method"],
            "rank": int(row["rank"]),
            "candidate_id": int(row["candidate_id"]),
            "candidate_name": row["candidate_name"],
            **result,
        }
        f_out.write(json.dumps(record) + "\n")
        f_out.flush()

        if i % 25 == 0:
            print(f"  {i:,} / {len(long_df):,} pairs processed")

print("\nDone judging.")

  0 / 4,040 pairs processed
  25 / 4,040 pairs processed
  50 / 4,040 pairs processed
  75 / 4,040 pairs processed
  100 / 4,040 pairs processed
  125 / 4,040 pairs processed
  150 / 4,040 pairs processed
  175 / 4,040 pairs processed
  200 / 4,040 pairs processed
  225 / 4,040 pairs processed
  250 / 4,040 pairs processed
  275 / 4,040 pairs processed
  300 / 4,040 pairs processed
  325 / 4,040 pairs processed
  350 / 4,040 pairs processed
  375 / 4,040 pairs processed
  400 / 4,040 pairs processed
  425 / 4,040 pairs processed
  450 / 4,040 pairs processed
  475 / 4,040 pairs processed
  500 / 4,040 pairs processed
  525 / 4,040 pairs processed
  550 / 4,040 pairs processed
  575 / 4,040 pairs processed
  600 / 4,040 pairs processed
  625 / 4,040 pairs processed
  650 / 4,040 pairs processed
  675 / 4,040 pairs processed
  700 / 4,040 pairs processed
  725 / 4,040 pairs processed
  750 / 4,040 pairs processed
  775 / 4,040 pairs processed
  800 / 4,040 pairs processed
  825 / 4,040 p

## 5. Aggregate into the two headline metrics, per method

Both metrics are **macro-averaged**: computed per anchor first, then averaged across anchors, so an anchor with 3 candidates is not out-weighted by one with a full 10. **Precision@10** is confirmed substitutes / candidates judged for that anchor. **Novel-substitute rate** is, among that anchor's confirmed substitutes, the fraction sitting beyond Step 1's own top-10 (rank > 10) — the genuinely new signal the trained space found — averaged over anchors that had at least one confirmed substitute.

In [5]:
# ---- Step 1 embeddings for the novel-substitute (rank > 10) check ----
p1_emb = np.load(STEP1_DIR / "embeddings.npy").astype(np.float32)
p1_emb = p1_emb / np.clip(np.linalg.norm(p1_emb, axis=1, keepdims=True), 1e-12, None)
with open(STEP1_DIR / "product_id_to_index.json") as f:
    p1_pid_to_idx = {int(pid): row for pid, row in json.load(f).items()}

_rank_cache = {}


def p1_rank_vec(anchor_id):
    """Cached Step 1 rank vector for an anchor (rank 0 = the anchor itself, 1 = its
    #1 neighbour). None if the anchor is not in the Step 1 catalogue."""
    anchor_id = int(anchor_id)
    if anchor_id not in _rank_cache:
        if anchor_id not in p1_pid_to_idx:
            _rank_cache[anchor_id] = None
        else:
            sims = p1_emb @ p1_emb[p1_pid_to_idx[anchor_id]]
            ranks = np.empty(len(sims), dtype=int)
            ranks[np.argsort(-sims)] = np.arange(len(sims))
            _rank_cache[anchor_id] = ranks
    return _rank_cache[anchor_id]


def is_novel(anchor_id, candidate_id):
    """True if candidate sits beyond Step 1's own top-10 for this anchor (rank > 10);
    None if either product is missing from the Step 1 catalogue."""
    ranks = p1_rank_vec(anchor_id)
    cid = int(candidate_id)
    if ranks is None or cid not in p1_pid_to_idx:
        return None
    return int(ranks[p1_pid_to_idx[cid]]) > 10


# ---- Load this run's judgments, scoped to the anchors + methods judged here ----
results_df = pd.DataFrame([json.loads(l) for l in open(RESULTS_PATH)])
results_df = results_df[results_df["anchor_id"].isin(judged_anchors)
                        & results_df["method"].isin(METHODS)].reset_index(drop=True)
clean = results_df[~results_df["parse_error"]].copy()

# ---- Macro-average: per anchor first, then average across anchors ----
summary_rows = []
for method in METHODS:
    m = clean[clean["method"] == method]
    per_anchor_prec = []
    per_anchor_novel = []
    for aid, grp in m.groupby("anchor_id"):
        confirmed = grp[grp["is_substitute"] == True]
        per_anchor_prec.append(len(confirmed) / len(grp))               # Precision@10 for this anchor
        novs = [is_novel(aid, cid) for cid in confirmed["candidate_id"]]
        novs = [x for x in novs if x is not None]
        if novs:                                                        # only anchors with confirmed subs
            per_anchor_novel.append(sum(novs) / len(novs))
    summary_rows.append({
        "method": method,
        "n_anchors_judged": int(m["anchor_id"].nunique()),
        "n_pairs_judged": int(len(m)),
        "precision_at_10": round(float(np.mean(per_anchor_prec)), 4) if per_anchor_prec else None,
        "novel_substitute_rate": round(float(np.mean(per_anchor_novel)), 4) if per_anchor_novel else None,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

summary = {
    "macro_average": True,
    "eval_sample_size": EVAL_SAMPLE_SIZE,
    "n_parse_errors": int(results_df["parse_error"].sum()),
    "methods": summary_rows,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved {SUMMARY_PATH.name}")

        method  n_anchors_judged  n_pairs_judged  precision_at_10  novel_substitute_rate
constant_theta               101            1010           0.2465                 0.7970
   pct_dropoff               101            1010           0.1723                 0.8162
       kneedle               101            1010           0.2564                 0.7410
     disparity               101            1010           0.2089                 0.7483

Saved llm_judge_summary.json


## 6. Judge reliability spot-check

30 sampled judged pairs for a manual read-through, deliberately oversampling organic/certification pairs (at least 10 if that many exist) since that was the exact failure mode just corrected — a flat random draw might not catch enough of them. Each entry shows the anchor, candidate, method, the judge's reasoning, and its resolved verdict.

In [6]:
import re

CERT_PAT = re.compile(r"organic|natural|cage[- ]?free", re.IGNORECASE)


def _is_cert(row):
    return bool(CERT_PAT.search(str(row["anchor_name"])) or CERT_PAT.search(str(row["candidate_name"])))


spot_rng = np.random.default_rng(0)

# results_df (from §5) holds this run's judgments. Oversample organic/cert pairs.
sp = results_df.reset_index(drop=True)
cert_idx = [i for i in sp.index if _is_cert(sp.loc[i])]
cert_set = set(cert_idx)
other_idx = [i for i in sp.index if i not in cert_set]

TARGET = 30
n_cert = min(10, len(cert_idx))
pick = list(spot_rng.choice(cert_idx, size=n_cert, replace=False)) if n_cert else []
n_other = min(TARGET - len(pick), len(other_idx))
pick += list(spot_rng.choice(other_idx, size=n_other, replace=False)) if n_other else []
# Top up with any remaining cert pairs if still short of TARGET.
if len(pick) < TARGET:
    leftover = [i for i in cert_idx if i not in set(pick)]
    extra = min(TARGET - len(pick), len(leftover))
    if extra:
        pick += list(spot_rng.choice(leftover, size=extra, replace=False))

n_cert_in_pick = sum(1 for i in pick if i in cert_set)


def _verdict_txt(v):
    return "SUBSTITUTE" if v is True else ("NOT-a-substitute" if v is False else "PARSE_ERROR")


with open(SPOTCHECK_PATH, "w") as f:
    f.write(f"LLM judge reliability spot-check — {len(pick)} pairs "
            f"({n_cert_in_pick} organic/certification-oversampled)\n")
    f.write("=" * 92 + "\n\n")
    for j, i in enumerate(pick, 1):
        r = sp.loc[int(i)]
        f.write(f"[{j:>2}] method={r['method']}   (Step 1 novel={is_novel(r['anchor_id'], r['candidate_id'])})\n")
        f.write(f"     anchor    : {r['anchor_name']} ({r['anchor_id']})\n")
        f.write(f"     candidate : {r['candidate_name']} ({r['candidate_id']})\n")
        f.write(f"     reason    : {r['reason']}\n")
        f.write(f"     directional={r['is_directional_pair']}  restricted_item={r['restricted_item']}  "
                f"raw_is_substitute={r['raw_is_substitute']}\n")
        f.write(f"     VERDICT   : {_verdict_txt(r['is_substitute'])}\n\n")

print(f"Wrote {len(pick)} spot-check pairs to {SPOTCHECK_PATH.name} "
      f"({n_cert_in_pick} organic/certification pairs)")

Wrote 30 spot-check pairs to llm_judge_spotcheck.txt (10 organic/certification pairs)
